###0. Import Necessary Dependencies for Datasets and Experiments

In [ ]:
!pip install -qqq arize arize-phoenix

In [ ]:
from arize.experimental.datasets.core.client import ArizeDatasetsClient
from typing import Any, Dict

from arize.experimental.datasets import ArizeDatasetsClient
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    Evaluator,
)
from arize.experimental.datasets.utils.constants import GENERATIVE
from openai import OpenAI

import pandas as pd
from dotenv import load_dotenv

import os
import requests

load_dotenv()

HOST = os.getenv("ARIZE_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")

In [ ]:
def create_dataset(api_key: str, name: str, space_id: str, examples):
    """
    Create a new dataset using the Arize API.
    
    Args:
        api_key: Arize API key
        name: Name of the dataset
        spaceId: Space ID where the dataset will be created
        examples: Either a pandas DataFrame or a list of example dictionaries to include in the dataset
    
    Returns:
        dict: Response from the API containing the created dataset information
    """

    rows = []

    for _, row in examples.iterrows():
        row_dict = {}
        for column in examples.columns:
            row_dict[column] = row[column]
        rows.append(row_dict)


    url = f"{HOST}/v2/datasets"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "name": name,
        "spaceId": space_id,
        "examples": rows
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()  # Raise an exception for bad status codes
    return response.json()

In [ ]:
df = pd.read_csv("../datasets/sample.csv")

df.head()

In [ ]:
dataset = create_dataset(API_KEY, "sample", SPACE_ID, df)

In [54]:
dataset

{'createdAt': '2025-12-17T20:15:57.513249Z',
 'id': 'RGF0YXNldDo4OmdyWmY=',
 'name': 'sample_v2',
 'spaceId': 'U3BhY2U6MTpiM2xN',
 'updatedAt': '2025-12-17T20:15:57.513249Z',
 'versions': [{'createdAt': '2025-12-17T20:15:57.515292Z',
   'datasetId': 'RGF0YXNldDo4OmdyWmY=',
   'id': 'RGF0YXNldFZlcnNpb246ODpiaWhR',
   'name': '2025-12-17 20:15:57',
   'updatedAt': '2025-12-17T20:15:57.513249Z'}]}

In [ ]:
import requests
import pandas as pd

def get_dataset(api_key: str, dataset_id: str):
    url = f"{HOST}/v2/datasets/{dataset_id}/examples?limit=50"
    headers = {
        "Authorization": f"Bearer {api_key}"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return pd.DataFrame(data['examples'])

###1. Get the dataset from Arize

In [ ]:
df = get_dataset(API_KEY, dataset["id"])
df